# Data Wrangling and Visualization Assignment

This notebook documents the workflow for parsing Wikipedia data about highest-grossing films, cleaning the extracted records, loading them into PostgreSQL, and exporting JSON for a static website.

## 1. Imports and configuration

The project uses requests and BeautifulSoup for scraping, psycopg for PostgreSQL, and the local `src` package for reusable ETL logic.

In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import settings
from src.wikipedia_parser import fetch_html, parse_highest_grossing_table, build_dataset
from src.database import get_connection, initialize_schema, upsert_films
from src.pipeline import export_json

settings

## 2. Fetch the Wikipedia source

The HTML snapshot is saved locally so the parsing step can be repeated without repeatedly downloading the page while iterating on the notebook.

In [ ]:
html = fetch_html(save_copy=True)
len(html)

## 3. Parse the highest-grossing films table

This step extracts the rank, title, release year, worldwide gross, and the film detail URL from the main Wikipedia table.

In [ ]:
base_records = parse_highest_grossing_table(html)
base_records[:3]

## 4. Enrich the records

To satisfy the assignment requirements, the pipeline visits each film page and extracts director and country information from the infobox.

In [ ]:
dataset = build_dataset()
len(dataset), [record.to_dict() for record in dataset[:3]]

## 5. Load records into PostgreSQL

Make sure PostgreSQL is running and your `.env` file is configured before executing the next cell.

In [ ]:
with get_connection(settings) as connection:
    initialize_schema(connection)
    rows_loaded = upsert_films(connection, [record.to_dict() for record in dataset])
rows_loaded

## 6. Export JSON for the website

The frontend reads `docs/data/films.json` directly, which makes it compatible with static hosting on GitHub Pages.

In [ ]:
export_json([record.to_dict() for record in dataset], project_root / 'docs' / 'data' / 'films.json')
(project_root / 'docs' / 'data' / 'films.json').exists()

## 7. Next steps

- Verify the exported `docs/data/films.json` file in the browser.
- Publish the `docs/` folder through GitHub Pages.
- Submit the notebook, repository link, and deployed page URL.